# 統計学特講

## 02 Pythonによるデータ分析

### 目的
- データの読み込み方法を学ぶ
- データの前処理
- データの統計処理

### 前準備

In [ ]:
import pandas as pd              # データフレーム関連のおまじない
import numpy as np               # 数値計算関連のおまじない
import scipy.stats as st         # 統計関連のおまじない
import statsmodels.api as sm     # 統計関連のおまじない
import matplotlib.pyplot as plt  # グラフ描画関連のおまじない

# データ置き場のディレクトリ指定
data_dir = "https://raw.githubusercontent.com/tnarumi/lecture-statistics/main/data/"

### 2-1 1変量データ

#### CSVデータをデータフレームに読み込み

In [ ]:
data_dice10 = data_dir + "dice_10.csv"
df_dice10 = pd.read_csv(data_dice10)

#### データの確認

##### 先頭の確認：括弧の中を省略すると最初の5個が表示される

In [ ]:
df_dice10.head()

列の名称(列の上部にある文字)を**ラベル**，行を表す数字を**インデックス**という．

##### 先頭の確認：括弧の中に数字を入れるとその数字だけ先頭から表示される

In [ ]:
df_dice10.head(10)

##### 末尾の確認

In [ ]:
df_dice10.tail()

##### データ数の確認

In [ ]:
df_dice10.size

#### 基本統計量の計算

##### 代表値

In [ ]:
print("平均:", df_dice10["dice"].mean())
print("中央値:", df_dice10["dice"].median())
print("最頻値:", df_dice10["dice"].mode())

##### データの散らばり

In [ ]:
print("分散:", df_dice10["dice"].var()) 
print("標準偏差:", df_dice10["dice"].std(ddof=0)) 

print("分散の平方根:", df_dice10["dice"].var() ** 0.5)

##### 基本統計量の一覧表示

In [ ]:
df_dice10["dice"].describe()

#### 可視化

##### ヒストグラム

In [ ]:
df_dice10["dice"].hist()

##### 箱ひげ図

In [ ]:
df_dice10["dice"].plot.box()

#### 信頼区間

In [ ]:
# 標準誤差（standard error; se）
se_d = df_dice10["dice"].std(ddof=1) / np.sqrt(len(df_dice10["dice"])) # std(ddof=1)は不偏分散を用いるための引数

# t分布を用いた95%信頼区間の計算
st.t.interval(0.95, df=len(df_dice10["dice"])-1, loc=df_dice10["dice"].mean(), scale=se_d)

### 2-2 1変量データ（練習用）

#### 1000個のデータを使って各種統計量の導出とヒストグラムの可視化

課題：下記内容を確認しましょう．
1. 平均を求めよ
2. 標準偏差を求めよ
3. ヒストグラムを描け
4. 箱ひげ図を描け
5. 統計量は理論値と比べてどうか

##### データの読み込み

In [ ]:
data_dice1000 = data_dir + "dice_1000.csv"
df_dice1000 = pd.read_csv(data_dice1000)

In [ ]:
# ここにコードを各自で追加してください


### 2-3 多変量データの前処理

#### データの読み込み

In [ ]:
data_bm1000 = data_dir + "body_measurements_1000.csv"
df_bm1000 = pd.read_csv(data_bm1000)
df_bm1000.head(15)

#### データの確認

##### データフレームの概要を確認

In [ ]:
df_bm1000.info()

null，NaN，na はデータが存在しないことを表す（0とは異なることに注意）

#### 前処理（欠損値）

##### 欠損確認

In [ ]:
df_bm1000.isna().sum()

##### 欠損処理

In [ ]:
df_bm1000_dropna = df_bm1000.dropna()
df_bm1000_dropna.info()

欠損があったデータは他のラベルのデータも使用しないことが原則

#### 前処理（外れ値）

##### ヒストグラム

In [ ]:
df_bm1000_dropna.hist(column="height_cm") # df_bm1000_dropna["height_cm"].hist() と同じ意味
df_bm1000_dropna.hist(column="weight_kg")
df_bm1000_dropna.hist(column="sleep_hours")
df_bm1000_dropna.hist(column="study_hours")

##### 各種統計量

In [ ]:
df_bm1000_dropna.describe()

##### 上位／下位のデータを確認

In [ ]:
df_bm1000_dropna.nlargest(10, "height_cm")

In [ ]:
df_bm1000_dropna.nlargest(10, "weight_kg")

In [ ]:
df_bm1000_dropna.nlargest(10, "sleep_hours")

In [ ]:
df_bm1000_dropna.nlargest(10, "study_hours")

In [ ]:
df_bm1000_dropna.nsmallest(10, "height_cm")

In [ ]:
df_bm1000_dropna.nsmallest(10, "weight_kg")

In [ ]:
df_bm1000_dropna.nsmallest(10, "sleep_hours")

In [ ]:
df_bm1000_dropna.nsmallest(10, "study_hours")

##### 条件設定により外れ値を削除

In [ ]:
df_bm1000_clean = df_bm1000_dropna[
  (df_bm1000_dropna["height_cm"] <= 250) &
  (df_bm1000_dropna["height_cm"] >=  50) &
  (df_bm1000_dropna["weight_kg"] <= 200) &
  (df_bm1000_dropna["weight_kg"] >=  50) 
  ]
df_bm1000_clean.describe()

プログラムコードで処理することにより，後からどういった条件でデータを削除したのかがわかる

意図的にデータを改変するのは研究不正になるので厳禁 → 外れ値は，明らかにおかしいもののみを外すこと．

### 2-4 多変量データの解析

#### 散布図によるデータの確認

In [ ]:
df_bm1000_clean.plot.scatter(x="height_cm", y="weight_kg") # pandasの散布図描画機能を使う方法

#### 共分散

In [ ]:
df_bm1000_clean[["height_cm", "weight_kg"]].cov()

#### 相関係数

In [ ]:
df_bm1000_clean[["height_cm","weight_kg"]].corr()

#### 線形単回帰分析

##### 解析

In [ ]:
x = sm.add_constant(df_bm1000_clean["height_cm"]) # 定数項を追加するための関数。定数項がないと、回帰直線は原点を通ることになる。
y = df_bm1000_clean["weight_kg"]
model = sm.OLS(y, x).fit()
model.params # 回帰係数の表示。定数項はconst、傾きはheight_cmに対応する値

##### 可視化（matplotlibを使用）

In [ ]:
plt.scatter(df_bm1000_clean["height_cm"], df_bm1000_clean["weight_kg"])
plt.plot(df_bm1000_clean["height_cm"], model.predict(x), color="red") # 回帰直線の描画

plt.xlabel("height_cm")
plt.ylabel("weight_kg")
plt.show()

#### 信頼区間

In [ ]:
#標準誤差
se_h = df_bm1000_clean["height_cm"].std(ddof=1) / np.sqrt(len(df_bm1000_clean["height_cm"]))

# 正規分布を用いた95%信頼区間の計算
st.norm.interval(0.95, loc=df_bm1000_clean["height_cm"].mean(), scale=se_h)

### 2-5 多変量データの解析（練習用）

#### 前処理が済んだデータフレーム（df_bm1000_clean）に対して，相関係数の計算や散布図の可視化をしなさい

In [ ]:
# ここにコードを各自で追加してください
